# Mass Spectrometry Analysis of Extracellular Vesicles Exploration with `mlcroissant`
This notebook demonstrates step-by-step how to load and explore the FAIR^2 dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library, referencing all scientific data entities (record sets, fields, columns) by their `@id` values only.

### Dataset Source
The original dataset is described in [FAIR\(^2\)](https://sen.science/doi/10.71728/senscience.vq4a-28xa) and the Croissant schema URL is:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Set Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset object
dataset = mlc.Dataset(croissant_url)

# Print main metadata properties
print(f"Dataset loaded: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets and their fields/columns, referencing all by their `@id` only.

In [ ]:
# Access Croissant entities
print("\nAvailable Record Sets and their Fields/Columns:")

entity_types = [
    ("record_sets", dataset.metadata.record_sets()),
    ("fields", dataset.metadata.fields()),
    ("columns", dataset.metadata.columns()),
    ("files", dataset.metadata.files()),
]
# Display only non-empty types
for name, values in entity_types:
    values = list(values)
    if values:
        print(f"\nFound {len(values)} {name.replace('_',' ')}:")
        for item in values:
            print(f"  @id: {item.id} | name: {getattr(item, 'name', None)}")

# Let's list record sets and their related fields
record_sets = list(dataset.metadata.record_sets())

for rset in record_sets:
    print(f"\nRecord Set @id: {rset.id} | name: {rset.name}")
    for field in rset.fields:
        print(f"    Field @id: {field.id} | name: {field.name}")

## 3. Data Extraction
Load records from selected record sets into a pandas DataFrame. Each data entity is referenced by its `@id` as per Croissant.

In [ ]:
# Select all record_set @id's for extraction
record_set_ids = [rset.id for rset in dataset.metadata.record_sets()]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nRecord set @id: {record_set_id}")
        print("Columns:", df.columns.tolist())
        display(df.head())
    else:
        print(f"\nNo records found for record_set: {record_set_id}")
# Pick the first record set id for EDA below
if record_set_ids:
    main_record_set_id = record_set_ids[0]
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply data processing with references by `@id`. Typical operations: filter records, normalize numeric fields, group by attributes.

In [ ]:
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    # Choose a numeric field by @id or fallback to the first suitable column
    numeric_field_id = None
    for field in dataset.metadata.fields():
        if field.id in df.columns and field.data_type in ('Float', 'Integer', 'Number'):
            numeric_field_id = field.id
            break
    if numeric_field_id is None:
        numeric_candidates = df.select_dtypes(include='number').columns
        if len(numeric_candidates):
            numeric_field_id = numeric_candidates[0]
    if numeric_field_id is not None:
        print(f"Using numeric field '@id': {numeric_field_id}")
        # Show basic stats
        print(df[numeric_field_id].describe())
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nRecords with '{numeric_field_id}' > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())/filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered rows:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Try grouping by a categorical field by @id
        group_field_id = None
        for field in dataset.metadata.fields():
            if field.id in df.columns and df[field.id].dtype == 'object' and field.id != numeric_field_id:
                group_field_id = field.id
                break
        if group_field_id:
            print(f"\nGrouped by '{group_field_id}':")
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            display(grouped.head())
        else:
            print("No suitable categorical field for grouping was found.")
    else:
        print("No numeric field found to process.")
else:
    print("No main record set with data found.")

## 5. Visualization
Visualize distribution of a numeric field and its grouping by categorical fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field_id is not None and main_record_set_id in dataframes:
    plt.figure(figsize=(6,4))
    df = dataframes[main_record_set_id]
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
    # If there is a suitable group field
    if group_field_id is not None:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by group {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion
- We loaded a mass spectrometry dataset defined by a Croissant schema and explored its overall structure via `@id` references.
- The data was extracted, a key numeric field was filtered and normalized, and distributions were visualized.
- You can adapt the EDA and visualizations to target specific biological questions or use `@id` values to reference particular columns relevant to your analysis, enabling clear provenance as per FAIR/FAIR\(^2\) standards.